# Input file preparation and output evaluation for template matching

This user guide covers the cryoCAT commands to prepare inputs required to run 3D template matching with GAPSTOP [[1]](#ref1), [[2]](#ref2) and to extract the particles for the output scores volumes. This page does not cover the details on how to set up a template matching job, it rather provides an overview of the functions that integrate in the pipeline and are available in cryoCAT. Please visit the official [documentation](https://bturo.pages.mpcdf.de/gapstop_tm/index.html# "GAPSTOP documentation") for details on all input files and how to run GAPSTOP.

The functions that are required for the integration with template matching pipelines can be accessed both from within a Python environment (e.g. in Jupyter notebooks) and via the Command Line Interface (CLI). Where applicable, we illustrate both.

## Input files preparation

cryoCAT can be used to prepare angle lists, wedge list and the mask that are used during template matching.

### Angle list

Angle lists provide the orientations that are sampled in the orientational space, expressed as Euler angles. Those can be generated with the function `geom.generate_angles()`. You can specify the desired search range and step for both the cone and the in-plane angles. The output will be a numpy.ndarray object with (N,3) dimensions, which will be saved to the specified file path containing one orientation (expressed as 3 Euler angles) in each line. The Euler angles convention follows the defaults ("zxz"), unless specified otherwise. For symmetric templates, the orientational space can be reduced using symmetry operators to test fewer orientations and spare computational time and resources. You can check which symmetry types are supported in cryoCAT in the [Symmetries basics](symmetries_basics.ipynb) user guide. Please be aware that for symmetric complexes, the template is assumed to be oriented according to the canonical frame, as defined in cryoCAT, otherwise the symmetry reduction will generate the wrong orientations.

The only mandatory parameters are `cone_angle` (the cone angle range) and `cone_sampling` (the angular step). By default, the function takes `inplane_angle=360` and `inplane_smapling=cone_sampling`, unless those parameters are explicitly set. 

#### Python code (Jupyter notebooks)

In [ ]:
from cryocat.utils.geom import generate_angles

# sample the whole sphere, using an angular step of 15 degrees for both the cone sampling 
# and the in-plane sampling, save the output list to "angle_list_c1.txt" (default C1 symmetry => no symmetry)
generate_angles(360,15, output_path="path/to/angle_list_c1.txt")

The output file generated by running the example cell above should contain 4776 orientations.

In [ ]:
# same as previous example, but with C13 symmetry
generate_angles(360,15, symmetry=13, output_path="path/to/angle_list_c13.txt")

Using _C13_ symmetry, the number of orientations are reduced by a factor of 12.

#### CLI command

The `geom.generate_angles()` function is accessible in the CLI as `cc_geom generate_angles`, which has the following arguments:

The equivalent commands to replicate the lists shown in the above examples via the CLI are indicated below:

### Mask preparation

The mask can be created using the `cryomask` module. Masks can be generated in a range of geometrical shapes and can be written to EM or MRC files, as illustrated in the ["Masks basics"](./masks_basics.ipynb) guide. You can visit that page for more details on how to prepare those using Python code. Here we will show some examples on how to create a mask directly from the command line using the `cc_cryomask` command. For an overview of the functions of `cryomask` available in the CLI, you can check:

As for template matching, you need to mind the following points:

- Your mask needs to have the same dimensions as your template. Please specify `--mask_size` accordingly;
- It is recommended to use a soft-edge mask. Make sure to add Gaussian blurring by specifying the `--gaussian` argument. By default, the blur will be done outward. This behavior can be changed by passing `--gaussian_outwards False`.


Below, we provide few examples for preparing different kind of masks directly from the command line via the CLI commands:

**Spherical mask**

**Cylindrical mask**

**Tight mask from the template**

Once the mask is ready, it is recommended to open it overlaid on the template to ensure that the latter is fully contained in the mask.

### Wedge list preparation

The wedge list contains metadata about the single tilt images making a tomogram as tilt angles, estimated defocus values, electron dose, etc. These files can be created with the `cryowedge` module as illustrated in the dedicated ["Wedge list basics"](./wedgelist_basics.ipynb) user guide. Please note that GAPSTOP requires a wedge list in starfile format. Below you can find both the Python function and the CLI command to create a wedge list with the proper format for template matching for a batch of tomograms forming a dataset. The output will be a single file with the information about all the tomograms. GAPSTOP will automatically filter the information for each tomogram.

#### Python code (Jupyter notebooks)

In [ ]:
from cryocat.core.cryowedge import *

_ = create_wedge_list_sg_batch("/path/to/tomo_list.txt",
                                2.176,
                                "/path/to/TS_$xxx/$xxx.mrc.mdoc",
                                tomo_dim=[4096, 4096, 1800],
                                ctf_file_format = "/path/to/TS_$xxx/$xxx_ctf_output.txt", 
                                ctf_file_type = "ctffind4",
                                output_path="/path/to/wedgelist.star",
                            )

#### Command line interface 

The CLI counterpart of the function above is `cc_cryowedge create_wedge_list_sg_batch`:

Example:

In [ ]:
cc_cryowedge create_wedge_list_sg_batch \
    --tomo_list /path/to/tomo_list.txt \
    --pixel_size 2.414 \
    --tlt_file_format '/path/to/TS_$xxx/$xxx.mrc.mdoc' \
    --tomo_dim /path/to/tomo_dims.txt \
    --ctf_file_format '/path/to/TS_$xxx/$xxx_gctf.star' \
    --ctf_file_type gctf \
    --dose_file_format '/path/to/TS_$xxx/$xxx.csv' \
    --output_path /path/to/wedge_list.star

N.B: 

- Paths with  some "$xxx" patterns (e.g. for `--tlt_file_format`) should be included within single inverted quotes;
- As for `--tomo_dim`, you can pass either a single file with information about tomogram dimension for all the tomograms, or a comma-separated triplet of the dimensions for all tomograms _x,y,z_ .

## Extract particles from scores volumes


Once a template match run is over, you need to extract a particle list from the candidate peaks. Those are stored in scores map, which contain the information about the coordinates in real space, and the angles map, which contain information about the orientations. You can conveniently use `tmana.scores_extract_particles()` to create a particle list from template matching outputs, either using Python or using the CLI command `cc_tmana scores_extract_particles`. This function operates on one single tomogram at a time, therefore you need to use it in a for loop if you want to run it on multiple tomograms. For symmetric complexes each particle gets a randomly selected symmetric copy of its orientation. The output particle list can be written out in any of the supported formats (EM, STAR, TBL).

#### Python code (Jupyter notebooks)

In [ ]:
from cryocat.analysis.tmana import *

# extract particles with the selected threshold from tomogram 51, no symmetry
extracted_motl = scores_extract_particles(
    "/path/to/scores_0_51.em", 
    "/path/to/angles_0_51.em", 
    angles_list="/path/to/angles_c1.txt", # angle list used for orientational sampling
    tomo_id=51,
    particle_diameter=23, #px
    scores_threshold=0.1, #threshold can be selected on a per-tomogram basis
    output_path="/path/to/output_motl.em"
)

#### Command line interface

In [ ]:
cc_tmana scores_extract_particles \
    --scores_map /path/to/scores_0_51.em \
    --angles_map /path/to/angles_0_51.em \
    --angles_list /path/to/angles_c1.txt  \
    --tomo_id 51 \
    --particle_diameter 12 \
    --scores_threshold 0.1 \
    --output_path /path/to/output_motl.em

## References
1. Turonova. GPU Accelerated Python-base Stopgap for Template Matching. (2024).
2. Cruz-León, S. et al. High-confidence 3D template matching for cryo-electron tomography. Nat Commun 15, 3992 (2024).


